# 07 — Phân tích Hình học Không gian Embedding và Trực quan hóa t-SNE (SV2)

Notebook này phân tích ma trận `item_embeddings.npy` (`37.583` sản phẩm $\times$ `64` chiều) của cả **3 mô hình (`GRU4Rec`, `NARM`, `SASRec`) $\times$ 2 seed (`2024`, `2025`)** để trả lời **Câu hỏi nghiên cứu 2 và 3** của đề tài:

> **Câu hỏi 2:** *Norm embedding ($\lVert e_i \rVert_2$) có tương quan với tần suất xuất hiện trong train (`train_frequency`) không, và mức độ tương quan Spearman $\rho$ (kèm p-value và khoảng tin cậy 95%) ở mỗi mô hình ra sao?*
>
> **Câu hỏi 3:** *Item cold (embedding chưa huấn luyện, chỉ để minh họa Option B) nằm ở đâu trong không gian embedding so với item warm cùng category?*

### Các bảng và biểu đồ xuất ra từ notebook này:
- `results/tables/norm_frequency.csv`, `results/tables/cosine_category.csv`, `results/tables/category_centroid.csv`
- `results/statistical_tests/spearman_results.csv`, `results/statistical_tests/cosine_tests.csv`
- `results/figures/norm_boxplot.png`, `results/figures/frequency_norm_scatter.png`, `results/figures/tsne_by_popularity.png`, `results/figures/tsne_by_category.png`

## 1. Thiết lập môi trường và đọc dữ liệu Embedding + Bảng Item-level

In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "BTL-K03-tong-quan-de-tai.md").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy thư mục gốc dự án BTL-K03.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from btl_k03.paths import (
    FIGURES_DIR,
    ITEM_LEVEL_DIR,
    PROCESSED_DIGI_DIR,
    RUNS_DIR,
    STAT_TESTS_DIR,
    TABLES_DIR,
    ensure_output_dirs,
)
from btl_k03.embedding_analysis import (
    compute_tsne_sample,
    sample_category_cosine_pairs,
    spearman_with_bootstrap_ci,
)
from btl_k03.utils import load_experiment_config, load_item_mapping

ensure_output_dirs()

COMMON_CFG = load_experiment_config("gru4rec")
MODELS = ["gru4rec", "narm", "sasrec"]
SEEDS = COMMON_CFG["seeds"]
item_mapping = load_item_mapping(PROCESSED_DIGI_DIR)

embeddings_dict = {}
item_tables_list = []

for model_name in MODELS:
    for seed in SEEDS:
        emb_path = RUNS_DIR / model_name / f"seed_{seed}" / "item_embeddings.npy"
        il_path = ITEM_LEVEL_DIR / f"{model_name}_seed{seed}.parquet"
        if not emb_path.is_file() or not il_path.is_file():
            raise FileNotFoundError(
                f"Chưa tìm thấy {emb_path} hoặc {il_path}. Hãy chạy các notebook 03, 04, 05 trước!"
            )
        embeddings_dict[(model_name, seed)] = np.load(emb_path)
        item_tables_list.append(pd.read_parquet(il_path))

all_item_df = pd.concat(item_tables_list, ignore_index=True)
print(f"Đã nạp {len(embeddings_dict)} ma trận item_embeddings.npy kích thước {embeddings_dict[(MODELS[0], SEEDS[0])].shape}")
print(f"Tổng số dòng trong all_item_df: {len(all_item_df):,}")

## 2. Phân tích Norm Embedding theo Nhóm Popularity và Tương quan Spearman (`train_freq ↔ norm`)

In [ ]:
norm_group_summary = (
    all_item_df.groupby(["model", "seed", "popularity_group"], sort=False)[
        ["embedding_norm", "distance_to_category_centroid", "cosine_to_category_centroid"]
    ]
    .agg(["count", "mean", "std", "median"])
    .reset_index()
)
norm_group_summary.columns = [
    "_".join([c for c in col if c]).strip("_") for col in norm_group_summary.columns.values
]
norm_group_summary.to_csv(TABLES_DIR / "norm_frequency.csv", index=False)
norm_group_summary.to_csv(TABLES_DIR / "category_centroid.csv", index=False)

print("=== BẢNG 1: THỐNG KÊ L2 NORM VÀ KHOẢNG CÁCH ĐẾN TÂM CATEGORY THEO NHÓM ITEM ===")
display(norm_group_summary)

# Tính tương quan Spearman giữa train_freq và embedding_norm (kèm p-value và 95% Bootstrap CI)
spearman_rows = []
n_boot = int(COMMON_CFG["embedding_geometry"].get("bootstrap_iterations", 1000))

for (model_name, seed), sub_df in all_item_df.groupby(["model", "seed"], sort=False):
    warm_sub = sub_df.loc[sub_df["is_trainable_warm"]]
    stat_warm = spearman_with_bootstrap_ci(
        warm_sub["train_freq"], warm_sub["embedding_norm"], n_bootstrap=n_boot, seed=seed
    )
    spearman_rows.append({
        "model": model_name.upper(),
        "seed": seed,
        "catalog_scope": "warm_only (29,936 items)",
        **stat_warm,
    })

    stat_all = spearman_with_bootstrap_ci(
        sub_df["train_freq"], sub_df["embedding_norm"], n_bootstrap=n_boot, seed=seed
    )
    spearman_rows.append({
        "model": model_name.upper(),
        "seed": seed,
        "catalog_scope": "all_catalog_incl_cold (37,583 items)",
        **stat_all,
    })

spearman_df = pd.DataFrame(spearman_rows)
spearman_df.to_csv(STAT_TESTS_DIR / "spearman_results.csv", index=False)

print("\n=== BẢNG 2: TƯƠNG QUAN SPEARMAN GIỮA TẦN SUẤT TRAIN VÀ L2 NORM EMBEDDING ===")
display(spearman_df)

## 3. Phân tích Độ tương đồng Cosine cùng Category và khác Category (Warm vs. Cold)

In [ ]:
n_pairs = int(COMMON_CFG["embedding_geometry"].get("pair_sample_size_per_group", 50000))
cosine_summary_frames = []
cosine_test_rows = []

for (model_name, seed), emb in embeddings_dict.items():
    cos_df, test_dict = sample_category_cosine_pairs(
        embeddings=emb,
        item_mapping=item_mapping,
        n_pairs_per_group=n_pairs,
        seed=seed,
    )
    cos_df.insert(0, "seed", seed)
    cos_df.insert(0, "model", model_name.upper())
    cosine_summary_frames.append(cos_df)
    cosine_test_rows.append({"model": model_name.upper(), "seed": seed, **test_dict})

cosine_category_df = pd.concat(cosine_summary_frames, ignore_index=True)
cosine_tests_df = pd.DataFrame(cosine_test_rows)

cosine_category_df.to_csv(TABLES_DIR / "cosine_category.csv", index=False)
cosine_tests_df.to_csv(STAT_TESTS_DIR / "cosine_tests.csv", index=False)

print("=== BẢNG 3: ĐỘ TƯƠNG ĐỒNG COSINE TRUNG BÌNH THEO NHÓM CẶP CATEGORY ===")
display(cosine_category_df)

print("\n=== BẢNG 4: KIỂM ĐỊNH THỐNG KÊ (MANN-WHITNEY U & WELCH T-TEST) CÙNG VS. KHÁC CATEGORY ===")
display(cosine_tests_df)

## 4. Biểu đồ Boxplot Norm và Scatter `log10(train_freq) ↔ embedding_norm`

In [ ]:
first_seed = SEEDS[0]
seed_item_df = all_item_df.loc[all_item_df["seed"] == first_seed].copy()
seed_item_df["model_upper"] = seed_item_df["model"].str.upper()

# Hình 1: Boxplot L2 Norm theo nhóm Popularity cho cả 3 mô hình
fig, ax = plt.subplots(figsize=(11, 5.5))
sns.boxplot(
    data=seed_item_df,
    x="popularity_group",
    y="embedding_norm",
    hue="model_upper",
    order=["popular", "medium", "rare", "cold"],
    palette="Set2",
    showfliers=False,
    ax=ax,
)
ax.set_title(f"Phân bố L2 Norm của Item Embedding theo Nhóm Độ phổ biến (Seed {first_seed})")
ax.set_xlabel("Nhóm Item (Popular / Medium / Rare / Cold chưa huấn luyện)")
ax.set_ylabel("L2 Norm ||e_i||_2")
ax.legend(title="Mô hình")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "norm_boxplot.png", dpi=300)
plt.show()

# Hình 2: Scatter plot log10(train_freq) vs embedding_norm cho 3 mô hình
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), sharey=False)
group_colors = {"popular": "#d95f02", "medium": "#7570b3", "rare": "#1b9e77"}

for idx, m in enumerate(MODELS):
    warm_m = seed_item_df.loc[(seed_item_df["model"] == m) & (seed_item_df["is_trainable_warm"])].copy()
    warm_m["log10_freq"] = np.log10(warm_m["train_freq"])
    rho_val = spearman_df.loc[
        (spearman_df["model"] == m.upper())
        & (spearman_df["seed"] == first_seed)
        & (spearman_df["catalog_scope"].str.startswith("warm_only")),
        "rho",
    ].iloc[0]

    sns.scatterplot(
        data=warm_m.sample(n=min(6000, len(warm_m)), random_state=first_seed),
        x="log10_freq",
        y="embedding_norm",
        hue="popularity_group",
        hue_order=["popular", "medium", "rare"],
        palette=group_colors,
        alpha=0.45,
        s=18,
        ax=axes[idx],
    )
    axes[idx].set_title(f"{m.upper()} — Spearman ρ = {rho_val:.4f} (p < 1e-4)")
    axes[idx].set_xlabel("log10(Train Frequency)")
    axes[idx].set_ylabel("L2 Norm ||e_i||_2")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "frequency_norm_scatter.png", dpi=300)
plt.show()

## 5. Trực quan hóa t-SNE (Tô màu theo Nhóm Tần suất và theo Category, Đánh dấu Cold Item)

> **Cảnh báo giới hạn diễn giải t-SNE (Mục 10.6 & Mục 17 của đề tài):**
> - t-SNE chỉ ưu tiên bảo toàn cấu trúc lân cận cục bộ; khoảng cách giữa các cụm trên mặt phẳng 2D không phản ánh tuyến tính khoảng cách thực trong không gian 64 chiều gốc.
> - Mọi kết luận định lượng phải dựa vào bảng số đo L2 Norm, Cosine Similarity và khoảng cách tâm Category ở Mục 2 & Mục 3; hình t-SNE chỉ đóng vai trò minh họa trực quan.

In [ ]:
tsne_cfg = COMMON_CFG["embedding_geometry"]["tsne"]
SAMPLE_SIZE = int(tsne_cfg.get("sample_size", 8000))
PERPLEXITY = int(tsne_cfg.get("perplexity", 30))
TSNE_SEED = int(first_seed)

tsne_frames = {}
for m in MODELS:
    emb = embeddings_dict[(m, TSNE_SEED)]
    il_sub = all_item_df.loc[(all_item_df["model"] == m) & (all_item_df["seed"] == TSNE_SEED)]
    print(f"Đang chạy t-SNE cho {m.upper()} (n={SAMPLE_SIZE}, perplexity={PERPLEXITY}, seed={TSNE_SEED})...")
    tsne_frames[m] = compute_tsne_sample(
        embeddings=emb,
        item_table=il_sub,
        sample_size=SAMPLE_SIZE,
        perplexity=PERPLEXITY,
        seed=TSNE_SEED,
    )

# Hình 3: t-SNE tô màu theo bin tần suất (Popular / Medium / Rare / Cold)
fig, axes = plt.subplots(1, 3, figsize=(19, 6))
pop_palette = {"popular": "#e41a1c", "medium": "#377eb8", "rare": "#4daf4a", "cold": "#333333"}
pop_markers = {"popular": "o", "medium": "o", "rare": "o", "cold": "X"}

for idx, m in enumerate(MODELS):
    tdf = tsne_frames[m]
    sns.scatterplot(
        data=tdf,
        x="tsne_1",
        y="tsne_2",
        hue="popularity_group",
        style="popularity_group",
        hue_order=["popular", "medium", "rare", "cold"],
        palette=pop_palette,
        markers=pop_markers,
        alpha=0.60,
        s=22,
        ax=axes[idx],
    )
    axes[idx].set_title(
        f"{m.upper()} — t-SNE theo Popularity Bin\n(n={ len(tdf):,}, perplexity={PERPLEXITY}, seed={TSNE_SEED})"
    )
    axes[idx].set_xlabel("t-SNE 1 (Lưu ý: Khoảng cách cụm chỉ mang tính minh họa)")
    axes[idx].set_ylabel("t-SNE 2")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "tsne_by_popularity.png", dpi=300)
plt.show()

# Hình 4: t-SNE tô màu theo Top Category và đánh dấu riêng Cold Item
fig, axes = plt.subplots(1, 3, figsize=(19, 6))
for idx, m in enumerate(MODELS):
    tdf = tsne_frames[m]
    top_tdf = tdf.loc[tdf["category_display"] != "Other Categories"].copy()
    sns.scatterplot(
        data=top_tdf,
        x="tsne_1",
        y="tsne_2",
        hue="category_display",
        style="is_cold",
        markers={False: "o", True: "X"},
        palette="tab10",
        alpha=0.75,
        s=32,
        ax=axes[idx],
    )
    axes[idx].set_title(
        f"{m.upper()} — t-SNE theo Top Categories (X = Cold Item)\n(perplexity={PERPLEXITY}, seed={TSNE_SEED})"
    )
    axes[idx].set_xlabel("t-SNE 1 (Minh họa cục bộ; định lượng dựa vào bảng Cosine)")
    axes[idx].set_ylabel("t-SNE 2")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "tsne_by_category.png", dpi=300)
plt.show()